# 서울시 부동산 실거래가 정보 (아파트) 수집

- 데이터 출처: [서울시 부동산 실거래가 정보](https://data.seoul.go.kr/dataList/OA-21275/S/1/datasetView.do) (서울 열린데이터광장 Open API, `tbLnOpendataRtmsV`)
- 인증키는 프로젝트 폴더의 `.env` 파일에 `SEOUL_API_KEY`로 저장되어 있음
- 건물용도(`BLDG_USG`)가 "아파트"이고, 계약일(`CTRT_DAY`)이 `TARGET_YEAR`(2026)에 속하는 거래를 전부 수집

**참고**
- 이 API는 조회 조건(자치구, 법정동, 건축년도 등)이 URL 경로상 위치 인자로 전달되는 방식이라, 뒤쪽 인자인 `BLDG_USG`/`CTRT_DAY`만 쓰고 앞의 인자들을 비워두면 조건이 무시되고 전체 데이터가 반환된다(직접 테스트로 확인). 또한 `CTRT_DAY`는 특정 하루(YYYYMMDD)만 지정할 수 있어 "연도 전체"를 API 단에서 직접 필터링할 수는 없다.
- 따라서 접수연도(`RCPT_YR`, 첫 번째 위치 인자)로 대량 수집한 뒤, 파이썬에서 `BLDG_USG == "아파트"` 및 계약일(`CTRT_DAY`)이 `TARGET_YEAR`에 속하는 행만 걸러내는 방식을 사용한다.
- 접수연도와 계약연도가 어긋나는 경우가 있어(예: 접수연도 2026건 중 계약일이 2025년인 경우, 반대로 연말 계약분이 다음 해에 접수되는 경우) `RCPT_YR`는 `TARGET_YEAR`와 `TARGET_YEAR+1`(이미 시작된 경우)까지 받아온 뒤, 계약일 기준으로 최종 필터링한다.

## 응답 필드 설명

| No | 필드명 | 설명 |
|----|--------|------|
| 1  | RCPT_YR | 접수연도 |
| 2  | CGG_CD | 자치구코드 |
| 3  | CGG_NM | 자치구명 |
| 4  | STDG_CD | 법정동코드 |
| 5  | STDG_NM | 법정동명 |
| 6  | LOTNO_SE | 지번구분 |
| 7  | LOTNO_SE_NM | 지번구분명 |
| 8  | MNO | 본번 |
| 9  | SNO | 부번 |
| 10 | BLDG_NM | 건물명 |
| 11 | CTRT_DAY | 계약일 |
| 12 | THING_AMT | 물건금액(만원) |
| 13 | ARCH_AREA | 건물면적(㎡) |
| 14 | LAND_AREA | 토지면적(㎡) |
| 15 | FLR | 층 |
| 16 | RGHT_SE | 권리구분 |
| 17 | RTRCN_DAY | 취소일 |
| 18 | ARCH_YR | 건축년도 |
| 19 | BLDG_USG | 건물용도 |
| 20 | DCLR_SE | 신고구분 |
| 21 | OPBIZ_RESTAGNT_SGG_NM | 신고한 개업공인중개사 시군구명 |

In [6]:
import os
import json
import time
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("SEOUL_API_KEY")
SERVICE = "tbLnOpendataRtmsV"
BASE_URL = f"http://openapi.seoul.go.kr:8088/{API_KEY}/json/{SERVICE}"

TARGET_YEAR = 2026  # 계약일(CTRT_DAY) 기준으로 수집할 연도

In [7]:
def fetch_rtms_page(start_index: int, end_index: int, rcpt_yr: str) -> list[dict]:
    """실거래가 데이터를 접수연도(rcpt_yr) 기준, START_INDEX ~ END_INDEX 범위로 요청 (1회 최대 1000건)

    CTRT_DAY(계약일)는 위치 인자상 특정 하루(YYYYMMDD)만 지정 가능해 "연도 전체" 조회에는 쓸 수 없다.
    그래서 첫 번째 위치 인자인 RCPT_YR(접수연도)로 대량 수집하고, 계약일 기준 연도 필터링은
    호출 이후 파이썬에서 CTRT_DAY 컬럼으로 수행한다.
    """
    url = f"{BASE_URL}/{start_index}/{end_index}/{rcpt_yr}/"
    response = requests.get(url)
    response.encoding = "utf-8"  # 인코딩을 지정하지 않으면 한글이 깨짐
    data = json.loads(response.text)[SERVICE]

    if data["RESULT"]["CODE"] not in ("INFO-000", "INFO-200"):
        raise RuntimeError(data["RESULT"]["MESSAGE"])

    return data.get("row", [])

In [8]:
# 접수연도별로 전체 페이지를 끝까지 수집
# 연말 계약분이 다음 해에 접수되는 경우가 있어, 이미 시작된 해라면 TARGET_YEAR+1 접수분도 함께 받는다
PAGE_SIZE = 1000  # 서울 열린데이터광장 정책상 1회 요청 최대 1000건 (ERROR-336)
years = range(TARGET_YEAR, min(TARGET_YEAR + 1, pd.Timestamp.today().year) + 1)

all_rows = []
for year in years:
    start = 1
    while True:
        end = start + PAGE_SIZE - 1
        rows = fetch_rtms_page(start, end, str(year))
        if not rows:
            break

        all_rows.extend(rows)
        print(f"{year}년: {start}~{start + len(rows) - 1}건 수신 (누적 {len(all_rows)}건)")

        if len(rows) < PAGE_SIZE:
            break  # 해당 연도의 마지막 페이지
        start += PAGE_SIZE
        time.sleep(0.1)  # 서버 부담 완화

print(f"전체 수신 건수: {len(all_rows)}")

2026년: 1~1000건 수신 (누적 1000건)
2026년: 1001~2000건 수신 (누적 2000건)
2026년: 2001~3000건 수신 (누적 3000건)
2026년: 3001~4000건 수신 (누적 4000건)
2026년: 4001~5000건 수신 (누적 5000건)
2026년: 5001~6000건 수신 (누적 6000건)
2026년: 6001~7000건 수신 (누적 7000건)
2026년: 7001~8000건 수신 (누적 8000건)
2026년: 8001~9000건 수신 (누적 9000건)
2026년: 9001~10000건 수신 (누적 10000건)
2026년: 10001~11000건 수신 (누적 11000건)
2026년: 11001~12000건 수신 (누적 12000건)
2026년: 12001~13000건 수신 (누적 13000건)
2026년: 13001~14000건 수신 (누적 14000건)
2026년: 14001~15000건 수신 (누적 15000건)
2026년: 15001~16000건 수신 (누적 16000건)
2026년: 16001~17000건 수신 (누적 17000건)
2026년: 17001~18000건 수신 (누적 18000건)
2026년: 18001~19000건 수신 (누적 19000건)
2026년: 19001~20000건 수신 (누적 20000건)
2026년: 20001~21000건 수신 (누적 21000건)
2026년: 21001~22000건 수신 (누적 22000건)
2026년: 22001~23000건 수신 (누적 23000건)
2026년: 23001~24000건 수신 (누적 24000건)
2026년: 24001~25000건 수신 (누적 25000건)
2026년: 25001~26000건 수신 (누적 26000건)
2026년: 26001~27000건 수신 (누적 27000건)
2026년: 27001~28000건 수신 (누적 28000건)
2026년: 28001~29000건 수신 (누적 29000건)
2026년: 29001~300

In [9]:
# 건물용도가 "아파트"인 행만 추출
apt_rows = [row for row in all_rows if row["BLDG_USG"] == "아파트"]

df = pd.DataFrame(apt_rows)

# 숫자/날짜형 컬럼 변환
df["THING_AMT"] = pd.to_numeric(df["THING_AMT"], errors="coerce")  # 물건금액 (만원)
df["ARCH_YR"] = pd.to_numeric(df["ARCH_YR"], errors="coerce")      # 건축년도
df["CTRT_DAY"] = pd.to_datetime(df["CTRT_DAY"], format="%Y%m%d")   # 계약일

# 접수연도 기준으로 받아왔으므로, 계약일(CTRT_DAY) 기준 TARGET_YEAR에 속하는 행만 최종적으로 남김
df = df[df["CTRT_DAY"].dt.year == TARGET_YEAR]
df = df.sort_values("CTRT_DAY", ascending=False).reset_index(drop=True)

print(f"아파트 거래 건수 (계약일 {TARGET_YEAR}년): {len(df)}")
df.head()

아파트 거래 건수 (계약일 2026년): 49959


,RCPT_YR,CGG_CD,CGG_NM,STDG_CD,STDG_NM,LOTNO_SE,LOTNO_SE_NM,MNO,SNO,BLDG_NM,...,THING_AMT,ARCH_AREA,LAND_AREA,FLR,RGHT_SE,RTRCN_DAY,ARCH_YR,BLDG_USG,DCLR_SE,OPBIZ_RESTAGNT_SGG_NM
0,2026,11530,구로구,11000,온수동,1,대지,0154,0000,리치안풍산,...,47000,84.990,0.0,2.0,,,2009,아파트,중개거래,서울 구로구
1,2026,11260,중랑구,10500,망우동,1,대지,0479,0000,한일써너스빌리젠시1단지(101동),...,88500,84.740,0.0,19.0,,,2008,아파트,중개거래,서울 중랑구
2,2026,11740,강동구,10900,천호동,1,대지,0019,0001,우성,...,83807,64.530,0.0,7.0,,,1985,아파트,직거래,
3,2026,11305,강북구,10100,미아동,1,대지,0194,0002,엘리프미아역1단지,...,84000,59.997,0.0,23.0,분양권,,0,아파트,중개거래,서울 강북구
4,2026,11650,서초구,10100,방배동,1,대지,0866,0020,방배대우디오빌,...,29800,29.600,0.0,8.0,,,2005,아파트,중개거래,서울 동작구


In [19]:
# 단위면적당 가격 산출 (THING_AMT: 만원, ARCH_AREA: 제곱미터)
PYEONG_M2 = 3.305785  # 1평 = 3.305785 제곱미터

df["PRICE_PER_SQM"] = (df["THING_AMT"] / df["ARCH_AREA"]).round(1)                # 만원/제곱미터
df["PRICE_PER_PYEONG"] = (df["THING_AMT"] / df["ARCH_AREA"] * PYEONG_M2).round(1)  # 만원/평

df[["BLDG_NM", "ARCH_AREA", "THING_AMT", "PRICE_PER_SQM", "PRICE_PER_PYEONG"]].head()

,BLDG_NM,ARCH_AREA,THING_AMT,PRICE_PER_SQM,PRICE_PER_PYEONG
0,리치안풍산,84.990,47000,553.0,1828.1
1,한일써너스빌리젠시1단지(101동),84.740,88500,1044.4,3452.5
2,우성,64.530,83807,1298.7,4293.3
3,엘리프미아역1단지,59.997,84000,1400.1,4628.3
4,방배대우디오빌,29.600,29800,1006.8,3328.1


In [20]:
output_path = "data/seoul_apt_price_2026.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")

In [21]:
df = pd.read_csv('data/seoul_apt_price_2026.csv')


## 지오코딩 준비: 고유 지번주소 추출

같은 아파트 단지(같은 자치구·법정동·본번·부번)에서 여러 건 거래가 반복되므로, 전체 거래 건수가 아니라 **고유 지번 주소만** 지오코딩 대상으로 추출한다.

In [12]:
def format_jibun_address(row) -> str:
    """구/동/본번/부번으로 지번주소 문자열 생성 (부번이 0이면 생략)"""
    mno = str(int(row["MNO"]))  # 앞자리 0 제거 (예: "0057" -> "57")
    sno = str(int(row["SNO"]))
    if sno == "0":
        return f"서울특별시 {row['CGG_NM']} {row['STDG_NM']} {mno}"
    return f"서울특별시 {row['CGG_NM']} {row['STDG_NM']} {mno}-{sno}"

ADDR_KEY_COLS = ["CGG_NM", "STDG_NM", "MNO", "SNO"]

# BLDG_NM은 그룹별 대표값(첫 값)만 남겨서, 지번주소로 지오코딩이 실패했을 때 건물명 검색 fallback에 사용
unique_addr = df[ADDR_KEY_COLS + ["BLDG_NM"]].drop_duplicates(subset=ADDR_KEY_COLS).reset_index(drop=True)
unique_addr["JIBUN_ADDR"] = unique_addr.apply(format_jibun_address, axis=1)

print(f"전체 거래 건수: {len(df)}")
print(f"고유 지번 주소 수: {len(unique_addr)}")
unique_addr.head()

전체 거래 건수: 49959
고유 지번 주소 수: 5272


,CGG_NM,STDG_NM,MNO,SNO,BLDG_NM,JIBUN_ADDR
0,구로구,온수동,154,0,리치안풍산,서울특별시 구로구 온수동 154
1,중랑구,망우동,479,0,한일써너스빌리젠시1단지(101동),서울특별시 중랑구 망우동 479
2,강동구,천호동,19,1,우성,서울특별시 강동구 천호동 19-1
3,강북구,미아동,194,2,엘리프미아역1단지,서울특별시 강북구 미아동 194-2
4,서초구,방배동,866,20,방배대우디오빌,서울특별시 서초구 방배동 866-20


## 지오코딩 (Kakao Local API)

고유 지번주소(`unique_addr`)에 대해서만 Kakao 주소 검색 API로 위경도를 조회한다. 주소 검색이 실패하면(재개발·명칭 변경 등) 자치구+법정동+건물명으로 키워드 검색을 한 번 더 시도한다. 결과는 캐시 파일로 저장해 두어, 다시 실행할 때는 API를 재호출하지 않는다.

In [13]:
load_dotenv()  # .env를 나중에 수정한 경우에도 반영되도록 재호출 (커널 재시작 없이도 최신 키를 읽음)
KAKAO_API_KEY = os.getenv("KAKAO_API_KEY")
if not KAKAO_API_KEY:
    raise RuntimeError("KAKAO_API_KEY가 비어 있습니다. .env 파일을 확인하고 커널을 재시작하세요.")

KAKAO_HEADERS = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
KAKAO_ADDRESS_URL = "https://dapi.kakao.com/v2/local/search/address.json"
KAKAO_KEYWORD_URL = "https://dapi.kakao.com/v2/local/search/keyword.json"


def _kakao_get(url: str, query: str) -> list[dict]:
    """Kakao Local API 호출 공통 로직. 인증 실패 등 오류는 조용히 넘기지 않고 즉시 예외로 드러냄"""
    response = requests.get(url, headers=KAKAO_HEADERS, params={"query": query})
    if response.status_code != 200:
        raise RuntimeError(f"Kakao API 오류 (status={response.status_code}): {response.text}")
    return response.json().get("documents", [])


def geocode_address(address: str):
    """지번주소로 위경도 조회. 검색 결과가 없으면 (None, None)"""
    documents = _kakao_get(KAKAO_ADDRESS_URL, address)
    if documents:
        return float(documents[0]["y"]), float(documents[0]["x"])
    return None, None


def geocode_keyword(keyword: str):
    """건물명 등 키워드로 장소 검색하여 위경도 조회. 검색 결과가 없으면 (None, None)"""
    documents = _kakao_get(KAKAO_KEYWORD_URL, keyword)
    if documents:
        return float(documents[0]["y"]), float(documents[0]["x"])
    return None, None

In [14]:
GEOCODE_CACHE_PATH = "data/unique_addr_geocoded.csv"

if os.path.exists(GEOCODE_CACHE_PATH):
    unique_addr = pd.read_csv(GEOCODE_CACHE_PATH, encoding="utf-8-sig")
    print(f"캐시 로드: {GEOCODE_CACHE_PATH} ({len(unique_addr)}건)")
else:
    lats, lons, methods = [], [], []

    for i, row in unique_addr.iterrows():
        lat, lon = geocode_address(row["JIBUN_ADDR"])
        method = "address"

        if lat is None:
            keyword = f"{row['CGG_NM']} {row['STDG_NM']} {row['BLDG_NM']}"
            lat, lon = geocode_keyword(keyword)
            method = "keyword" if lat is not None else "fail"

        lats.append(lat)
        lons.append(lon)
        methods.append(method)

        if (i + 1) % 200 == 0:
            print(f"{i + 1}/{len(unique_addr)}건 처리")

        time.sleep(0.05)  # 서버 부담 완화

    unique_addr["LAT"] = lats
    unique_addr["LON"] = lons
    unique_addr["GEOCODE_METHOD"] = methods
    unique_addr.to_csv(GEOCODE_CACHE_PATH, index=False, encoding="utf-8-sig")
    print(f"캐시 저장: {GEOCODE_CACHE_PATH}")

print(unique_addr["GEOCODE_METHOD"].value_counts())

200/5272건 처리
400/5272건 처리
600/5272건 처리
800/5272건 처리
1000/5272건 처리
1200/5272건 처리
1400/5272건 처리
1600/5272건 처리
1800/5272건 처리
2000/5272건 처리
2200/5272건 처리
2400/5272건 처리
2600/5272건 처리
2800/5272건 처리
3000/5272건 처리
3200/5272건 처리
3400/5272건 처리
3600/5272건 처리
3800/5272건 처리
4000/5272건 처리
4200/5272건 처리
4400/5272건 처리
4600/5272건 처리
4800/5272건 처리
5000/5272건 처리
5200/5272건 처리
캐시 저장: data/unique_addr_geocoded.csv
GEOCODE_METHOD
address    5263
keyword       8
fail          1
Name: count, dtype: int64


In [15]:
# 지오코딩된 좌표를 지번주소 키(구/동/본번/부번) 기준으로 전체 거래 df에 병합
df = df.merge(unique_addr[ADDR_KEY_COLS + ["LAT", "LON"]], on=ADDR_KEY_COLS, how="left")

n_success = df["LAT"].notna().sum()
print(f"지오코딩 성공: {n_success} / {len(df)} ({n_success / len(df):.1%})")

df[df["LAT"].isna()][["CGG_NM", "STDG_NM", "BLDG_NM"]].drop_duplicates()

지오코딩 성공: 49958 / 49959 (100.0%)


,CGG_NM,STDG_NM,BLDG_NM
43683,동작구,사당동,현대주택조합


In [16]:
df

,RCPT_YR,CGG_CD,CGG_NM,STDG_CD,STDG_NM,LOTNO_SE,LOTNO_SE_NM,MNO,SNO,BLDG_NM,...,LAND_AREA,FLR,RGHT_SE,RTRCN_DAY,ARCH_YR,BLDG_USG,DCLR_SE,OPBIZ_RESTAGNT_SGG_NM,LAT,LON
0,2026,11530,구로구,11000,온수동,1,대지,154,0,리치안풍산,...,0.0,2.0,NaN,NaN,2009,아파트,중개거래,서울 구로구,37.495866,126.817576
1,2026,11260,중랑구,10500,망우동,1,대지,479,0,한일써너스빌리젠시1단지(101동),...,0.0,19.0,NaN,NaN,2008,아파트,중개거래,서울 중랑구,37.597335,127.093478
2,2026,11740,강동구,10900,천호동,1,대지,19,1,우성,...,0.0,7.0,NaN,NaN,1985,아파트,직거래,NaN,37.549768,127.138816
3,2026,11305,강북구,10100,미아동,1,대지,194,2,엘리프미아역1단지,...,0.0,23.0,분양권,NaN,0,아파트,중개거래,서울 강북구,37.629276,127.025459
4,2026,11650,서초구,10100,방배동,1,대지,866,20,방배대우디오빌,...,0.0,8.0,NaN,NaN,2005,아파트,중개거래,서울 동작구,37.486044,126.984201
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49954,2026,11470,양천구,10300,신월동,1,대지,1026,0,양천벽산블루밍1단지,...,0.0,4.0,NaN,NaN,2006,아파트,중개거래,서울 양천구,37.524360,126.839446
49955,2026,11380,은평구,10600,대조동,1,대지,9,19,모아아파트(103동),...,0.0,12.0,NaN,NaN,2015,아파트,중개거래,서울 은평구,37.610228,126.927766
49956,2026,11680,강남구,11500,수서동,1,대지,746,0,까치마을,...,0.0,3.0,NaN,20260309.0,1993,아파트,중개거래,서울 강남구,37.484734,127.087618
49957,2026,11260,중랑구,10600,신내동,1,대지,397,0,동성1,...,0.0,2.0,NaN,NaN,1993,아파트,중개거래,서울 중랑구,37.609116,127.097679


In [22]:
output_path = "data/seoul_apt_price_2026_geocoded.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"저장 완료: {output_path} ({len(df)}건)")

저장 완료: data/seoul_apt_price_2026_geocoded.csv (49959건)


### Folium 시각화

## Folium 시각화: 평당가격 분포

지번(위경도) 단위로 거래를 집계해 평균 평당가격(`PRICE_PER_PYEONG`)을 계산한 뒤, 값이 클수록 원이 커지고 색이 진해지도록 표시한다. 배경 지도는 CartoDB Positron(그레이톤).

In [2]:
import pandas as pd

seoul_apt_price_2026_geocoded=pd.read_csv('data/seoul_apt_price_2026_geocoded.csv', encoding='utf-8-sig')
seoul_apt_price_2026_geocoded

,RCPT_YR,CGG_CD,CGG_NM,STDG_CD,STDG_NM,LOTNO_SE,LOTNO_SE_NM,MNO,SNO,BLDG_NM,...,RGHT_SE,RTRCN_DAY,ARCH_YR,BLDG_USG,DCLR_SE,OPBIZ_RESTAGNT_SGG_NM,LAT,LON,PRICE_PER_SQM,PRICE_PER_PYEONG
0,2026,11530,구로구,11000,온수동,1,대지,154,0,리치안풍산,...,NaN,NaN,2009,아파트,중개거래,서울 구로구,37.495866,126.817576,553.0,1828.1
1,2026,11260,중랑구,10500,망우동,1,대지,479,0,한일써너스빌리젠시1단지(101동),...,NaN,NaN,2008,아파트,중개거래,서울 중랑구,37.597335,127.093478,1044.4,3452.5
2,2026,11740,강동구,10900,천호동,1,대지,19,1,우성,...,NaN,NaN,1985,아파트,직거래,NaN,37.549768,127.138816,1298.7,4293.3
3,2026,11305,강북구,10100,미아동,1,대지,194,2,엘리프미아역1단지,...,분양권,NaN,0,아파트,중개거래,서울 강북구,37.629276,127.025459,1400.1,4628.3
4,2026,11650,서초구,10100,방배동,1,대지,866,20,방배대우디오빌,...,NaN,NaN,2005,아파트,중개거래,서울 동작구,37.486044,126.984201,1006.8,3328.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49954,2026,11470,양천구,10300,신월동,1,대지,1026,0,양천벽산블루밍1단지,...,NaN,NaN,2006,아파트,중개거래,서울 양천구,37.524360,126.839446,1046.8,3460.6
49955,2026,11380,은평구,10600,대조동,1,대지,9,19,모아아파트(103동),...,NaN,NaN,2015,아파트,중개거래,서울 은평구,37.610228,126.927766,980.9,3242.8
49956,2026,11680,강남구,11500,수서동,1,대지,746,0,까치마을,...,NaN,20260309.0,1993,아파트,중개거래,서울 강남구,37.484734,127.087618,4094.1,13534.1
49957,2026,11260,중랑구,10600,신내동,1,대지,397,0,동성1,...,NaN,NaN,1993,아파트,중개거래,서울 중랑구,37.609116,127.097679,830.0,2743.7


In [3]:
ADDR_KEY_COLS = ["CGG_NM", "STDG_NM", "MNO", "SNO"]

In [7]:
import folium
import branca.colormap as cm
from dotenv import load_dotenv


# 지번(위경도) 단위로 집계 (거래 건수만큼 점을 찍으면 같은 위치에 과도하게 겹침)
map_df = (
    seoul_apt_price_2026_geocoded.dropna(subset=["LAT", "LON"])
    .groupby(ADDR_KEY_COLS, as_index=False)
    .agg(
        LAT=("LAT", "first"),
        LON=("LON", "first"),
        BLDG_NM=("BLDG_NM", "first"),
        PRICE_PER_PYEONG=("PRICE_PER_PYEONG", "mean"),   # 평당가격 평균 
        N_TRADES=("PRICE_PER_PYEONG", "size"),           # 거래건수 합계
    )
)

print(f"지도에 표시할 지번 수: {len(map_df)}")

지도에 표시할 지번 수: 5271


In [14]:
# 점 크기: 평당가격에 비례 (최소~최대 반지름으로 정규화)
MIN_RADIUS, MAX_RADIUS = 4, 20
price_min = map_df["PRICE_PER_PYEONG"].min()
price_max = map_df["PRICE_PER_PYEONG"].max()


def price_to_radius(price: float) -> float:
    ratio = (price - price_min) / (price_max - price_min)
    return MIN_RADIUS + ratio * (MAX_RADIUS - MIN_RADIUS)


# 색상: 평당가격이 높을수록 진하게
colormap = cm.LinearColormap(
    colors=["#fee8c8", "#fdbb84", "#e34a33", "#7f0000"],
    vmin=price_min,
    vmax=price_max,
    caption="평당가격 (만원/평)",
)

load_dotenv()  # .env를 나중에 수정한 경우에도 반영되도록 재호출
CARTO_API_KEY = os.getenv("CARTO_API_KEY")
if not CARTO_API_KEY:
    raise RuntimeError("CARTO_API_KEY가 비어 있습니다. .env 파일을 확인하고 커널을 재시작하세요.")

# CartoDB Positron(그레이톤) 배경지도. CARTO가 2026년 8월부터 래스터 베이스맵에 API 키를 요구하므로 쿼리 파라미터로 첨부
CARTODB_POSITRON_TILES = f"https://{{s}}.basemaps.cartocdn.com/rastertiles/light_all/{{z}}/{{x}}/{{y}}.png?key={CARTO_API_KEY}"
CARTODB_ATTR = (
    '&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> contributors '
    '&copy; <a href="https://carto.com/attributions">CARTO</a>'
)

m = folium.Map(
    location=[map_df["LAT"].mean(), map_df["LON"].mean()],
    zoom_start=11,
    tiles=CARTODB_POSITRON_TILES,
    attr=CARTODB_ATTR,
)

for _, row in map_df.iterrows():
    price = row["PRICE_PER_PYEONG"]
    folium.CircleMarker(
        location=[row["LAT"], row["LON"]],
        radius=price_to_radius(price),
        color="#a35e33",  # 외곽선: 연한 갈색 (연한 회색보다 시각적으로 더 잘 구분됨)
        weight=0.3,
        fill=True,
        fill_color=colormap(price),
        fill_opacity=0.5,
        popup=folium.Popup(
            f"{row['BLDG_NM']}<br>평당가격: {price:,.0f}만원<br>거래건수: {row['N_TRADES']}건",
            max_width=200,
        ),
    ).add_to(m)

colormap.add_to(m)
m

In [15]:
MAP_OUTPUT_PATH = "html/seoul_apt_price_2026_map.html"
os.makedirs(os.path.dirname(MAP_OUTPUT_PATH), exist_ok=True)
m.save(MAP_OUTPUT_PATH)
print(f"저장 완료: {MAP_OUTPUT_PATH}")

저장 완료: html/seoul_apt_price_2026_map.html
